# Phase 1 — curation report

Reads `data/curated/report_dataset.jsonl` (every sample processed, accepted or not) and breaks down what got filtered out and why. This is the evidence-of-rigor artifact for the README: not just "I cleaned the data", but how much was discarded and for which reasons.

Dependencies (`pandas`, `matplotlib`, `seaborn`) are in the `dev` dependency group in `pyproject.toml` — run `uv sync` rather than `pip install`ing them inline, so the notebook's environment stays in sync with the rest of the project.

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.data_curation.config import REPORT_PATH

sys.path.insert(0, str(Path.cwd().parent))

metadata = []
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        metadata.append(sample["metadata"])

df = pd.DataFrame(metadata)
total = len(df)
accepted = (df["status"] == "accepted").sum()
rejected = total - accepted

print(f"Samples analyzed: {total}")
print(f"Accepted: {accepted} ({accepted / total:.1%})")
print(f"Rejected: {rejected} ({rejected / total:.1%})")

## Accepted vs. rejected

In [ ]:
plt.pie(
    [accepted, rejected],
    labels=["Accepted", "Rejected"],
    autopct="%1.1f%%",
    colors=["#4CAF50", "#F44336"],
)
plt.title("Accepted vs. rejected")
plt.show()

## Rejection reasons

Status categories (`rejected_no_code`, `rejected_wrong_language`, `rejected_ast`, `rejected_lint`, `rejected_complexity`) are the meaningful grouping here — not the raw `error` text, which is mostly unique strings (different line numbers, different messages) and won't aggregate into anything readable.

In [ ]:
df_rejected = df[df["status"] != "accepted"]
order = df_rejected["status"].value_counts().index
sns.countplot(data=df_rejected, x="status", order=order)
plt.title("Rejection reasons")
plt.xticks(rotation=20)
plt.show()

## Example errors per rejection reason

A few representative examples per category, rather than a `value_counts()` on raw error text (which was the original approach — it fragments into near-unique buckets since e.g. two "invalid syntax" errors at different line numbers count as different strings).

In [ ]:
for status, group in df_rejected.groupby("status"):
    print(f"\n{status} ({len(group)} samples) — example errors:")
    sample_n = min(3, len(group))
    for err in group["error"].dropna().sample(sample_n, random_state=0):
        print(f"  - {err}")